# EnvSDD preprocessing

Everything `run_preprocessing.sh` does, as notebook cells. Run top to bottom.

**Where the data goes.** Nothing is uploaded. Clips are *downloaded* from the
HuggingFace datasets-server (`datasets-server.huggingface.co/rows`) onto whatever
machine runs this notebook, into `data/processed/<split>/`. On your laptop that is
the project folder. On Colab it is the throwaway VM disk, so the last cell zips the
result up and hands it back to you.

Safe to re-run at any point: clips already on disk are skipped.

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = "https://github.com/Neha-Jacob-8/environmental-sound-deepfake-detection.git"
    ROOT = Path("/content/speech")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "requests", "soundfile", "torch", "torchaudio",
                    "pandas", "numpy", "matplotlib"], check=True)
else:
    # this notebook lives in notebooks/, one level below the project root
    ROOT = Path.cwd()
    if not (ROOT / "src").exists():
        ROOT = ROOT.parent

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("project root :", ROOT)
print("running on   :", "Colab" if IN_COLAB else "local machine")
print("data lands in:", ROOT / "data" / "processed")

## 2. How much to fetch

`GROUPS` counts **source recordings**, not clips. One train/validation group is
5 clips (1 real + G01-G04); one test group is 8 clips (1 real + G01-G07).

Start with the pilot numbers to check the pipeline works, then switch to the full
subset. The defaults below are the full subset — about 9,900 clips, ~1.7 GB.

In [ ]:
TRAIN, VAL, TEST = 1200, 300, 300     # full subset  (~1.7 GB, tens of minutes)
# TRAIN, VAL, TEST = 40, 20, 20       # pilot        (~35 MB, a few minutes)

WORKERS = 8                            # parallel clip downloads
SEED    = 1337                         # keep fixed: a re-run then retries exactly the gaps

n_clips = TRAIN * 5 + VAL * 5 + TEST * 8
print(f"{n_clips} clips, roughly {n_clips * 128 / 1024:.0f} MB")

## 3. Fetch

This is the long cell. Progress prints per request. If the network drops, just run
the cell again — finished clips are skipped and the seeded plan retries the rest.

In [ ]:
from src.preprocessing.fetch_subset import build

for split, groups, chunks in [("train", TRAIN, 20),
                              ("validation", VAL, 10),
                              ("test", TEST, 10)]:
    print(f"\n{'=' * 60}\nfetching {split}\n{'=' * 60}", flush=True)
    build(split, groups, chunks, out_root="data",
          dry_run=False, seed=SEED, workers=WORKERS)

## 4. Verify and build the manifest

Checks every WAV is 16 kHz / 64,000 samples / mono / non-silent, drops anything
broken, and writes `data/metadata/manifest.csv`. That manifest is what every
downstream model reads, and it is **regenerated, not appended** — so run this
after every fetch.

In [ ]:
import importlib
from src.preprocessing import verify_subset
importlib.reload(verify_subset)

sys.argv = ["verify_subset"]          # add "--strict" to fail on any problem clip
try:
    verify_subset.main()
except SystemExit as e:
    print("exit code:", e.code)

## 5. What you ended up with

In [ ]:
import pandas as pd

m = pd.read_csv("data/metadata/manifest.csv")
print(m.groupby(["split", "generator"]).size().unstack(fill_value=0), "\n")
print("total clips  :", len(m))
print("source groups:", m.groupby("split").source_id.nunique().to_dict())
print("real / fake  :", m.label.value_counts().to_dict())
print("\nsource datasets per split:")
print(pd.crosstab(m.split, m.source_dataset))

In [ ]:
# actual bytes on disk, and where
!du -sh data/processed/* data/metadata

print("\nfilenames are <split>_<source_id>_<generator>.wav:")
for f in sorted(os.listdir("data/processed/test"))[:8]:
    print("   ", f)

## 6. Look at the actual model input

A source group is one real recording plus every fake derived from it — same
content, same length, so the only difference is the generator. That is exactly
the signal the detector has to pick up.

The grid below is **not** a decorative spectrogram: it is the `(1, 64, 251)`
tensor `EnvSDDDataset` hands the CNN, one clip per generator, plotted straight
from `ds[i]`. If something is wrong with the front end — silent clips, a bad
mel range, saturated dB — it shows up here.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from src.datasets.envsdd_dataset import EnvSDDDataset, N_MELS, HOP, SR

ds = EnvSDDDataset(split="test", mode="logmel")
gid = ds.df.source_id.iloc[0]
group = ds.df[ds.df.source_id == gid].sort_values("generator")

# --- log-Mel: exactly what the CNN sees, one clip per generator ---------
fig, axes = plt.subplots(2, 4, figsize=(17, 6), sharex=True, sharey=True)
vmin, vmax = -40, 40
for ax, i in zip(axes.ravel(), group.index):
    x, y = ds[i]                       # x is (1, N_MELS, frames)
    row = ds.df.iloc[i]
    im = ax.imshow(x[0].numpy(), origin="lower", aspect="auto", cmap="magma",
                   vmin=vmin, vmax=vmax,
                   extent=[0, x.shape[-1] * HOP / SR, 0, N_MELS])
    ax.set_title(f"{row.generator}  ({'real' if row.label == 0 else 'fake'})",
                 fontsize=11)
for ax in axes[-1]:
    ax.set_xlabel("time (s)")
for ax in axes[:, 0]:
    ax.set_ylabel("mel bin")
fig.colorbar(im, ax=axes, shrink=0.7, label="dB")
fig.suptitle(f"log-Mel input to the CNN - test source group {gid}, "
             f"same recording through 8 generators", fontsize=13)
plt.show()

# --- per-clip tensor stats ---------------------------------------------
print(f"{'generator':10s} {'shape':>16s} {'min':>8s} {'max':>8s} {'mean':>8s} {'std':>8s}")
for i in group.index:
    x, _ = ds[i]
    r = ds.df.iloc[i]
    print(f"{r.generator:10s} {str(tuple(x.shape)):>16s} "
          f"{x.min():8.2f} {x.max():8.2f} {x.mean():8.2f} {x.std():8.2f}")

# --- waveform: what AASIST sees ----------------------------------------
raw = EnvSDDDataset(split="test", mode="waveform")
real_i = group.index[group.generator.values == "REAL"][0]
fake_i = group.index[group.generator.values == "G06"][0]

fig, axes = plt.subplots(2, 1, figsize=(15, 4), sharex=True, sharey=True)
for ax, i, name in [(axes[0], real_i, "REAL"),
                    (axes[1], fake_i, "G06 (TangoFlux, unseen)")]:
    w = raw[i][0][0].numpy()
    ax.plot(np.arange(len(w)) / SR, w, lw=0.4)
    ax.set_ylabel("amp"); ax.set_title(name, fontsize=10, loc="left")
axes[-1].set_xlabel("time (s)")
plt.suptitle("waveform input to AASIST - peak-normalised at load time")
plt.tight_layout(); plt.show()

print("REAL:"); display(Audio(ds.df.iloc[real_i].path))
print("G06 (TangoFlux, unseen at train time):"); display(Audio(ds.df.iloc[fake_i].path))

## 7. Check the PyTorch Dataset loads

`EnvSDDDataset` reads the manifest and returns either raw waveforms (for AASIST)
or log-Mel spectrograms (for the CNN baseline). Peak normalisation happens here,
at load time, not on disk — so the raw clips stay available for the augmentation
experiments.

`seen_unseen_test_loaders` is the core generalisation experiment: G01-G04 vs
G05-G07, both drawn from the test split so they share the same real clips and the
same source-domain mix.

In [ ]:
from src.datasets.envsdd_dataset import EnvSDDDataset, seen_unseen_test_loaders

train_ds = EnvSDDDataset(split="train", mode="logmel")
print(train_ds.describe())
print("logmel   :", train_ds[0][0].shape)
print("waveform :", EnvSDDDataset(split="train", mode="waveform")[0][0].shape)
print("pos_weight for BCEWithLogitsLoss:", train_ds.class_weights().item())

seen, unseen = seen_unseen_test_loaders(mode="logmel", batch_size=32, num_workers=0)
print("\nseen   G01-G04:", len(seen.dataset), "clips")
print("unseen G05-G07:", len(unseen.dataset), "clips")

import torch
print("\ndevice available:",
      "mps" if torch.backends.mps.is_available()
      else "cuda" if torch.cuda.is_available() else "cpu")

## 8. Colab only — get the data off the VM

The Colab VM is wiped when the session ends, so save the subset somewhere first.
Two options: download the zip to your laptop, or drop it in Google Drive.

Skip this cell entirely when running locally — the data is already in your
project folder.

In [ ]:
if IN_COLAB:
    !zip -qr /content/envsdd_subset.zip data/processed data/metadata
    !du -sh /content/envsdd_subset.zip

    # Option A - download to your laptop
    from google.colab import files
    files.download("/content/envsdd_subset.zip")

    # Option B - save to Google Drive instead (comment out A, uncomment this)
    # from google.colab import drive
    # drive.mount("/content/drive")
    # !cp /content/envsdd_subset.zip /content/drive/MyDrive/
else:
    print("Running locally - data is already at", (ROOT / 'data' / 'processed'))

Then unzip it into the project root on your laptop:

```bash
cd /Users/nehajacob/Downloads/YourFolder/sem7/sem7college/projects/speech
unzip -o ~/Downloads/envsdd_subset.zip
python3 -m src.preprocessing.verify_subset
```

The paths in `manifest.csv` are relative (`data/processed/...`), so they resolve
correctly as long as you run from the project root.